# 고급 프롬프트 설계 (Advanced Prompt Designs)

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## 사전 준비 사항

- LLM(대규모 언어 모델)에 대한 이해:
 - LLM이 무엇이며 어떻게 동작하는지.
 - 다음 토큰을 반복적으로 예측하는 장치로서의 LLM.
 - LLM의 예측은 학습 데이터와의 유사성을 최대화하는 방향으로 이루어진다는 점.
- LLM 프롬프팅 경험:
 - 언어 모델에 "프롬프트를 준다"는 것의 의미. [추천 자료](https://cloud.google.com/vertex-ai/docs/generative-ai/learn/introduction-prompt-design).
 - [제로샷, 원샷, 퓨샷](https://cloud.google.com/vertex-ai/docs/generative-ai/learn/introduction-prompt-design#include-examples) 프롬프팅의 차이, 그리고 성능과 견고성을 극대화하는 데 퓨샷 프롬프팅이 왜 필수적인지에 대한 이해.
- Google Cloud Vertex LLM에 대한 기본적인 친숙도. [추천 자료](https://cloud.google.com/vertex-ai/docs/generative-ai/start/quickstarts/api-quickstart)

## 핵심 용어

일관성을 위해 이 노트북에서는 다음 용어를 특정한 의미로 사용합니다.

* **프롬프트(Prompt)**: 템플릿 형태의 LLM 호출. 템플릿에 어떤 값이 삽입되더라도 호출의 성능과 견고성이 극대화되도록 하는 기법으로 작성됩니다.
* **LLM 호출(LLM Call)**: LLM에 텍스트를 전달하는 것.
* **LLM 응답(LLM Response)**: LLM이 예측한 텍스트, 즉 LLM 호출을 했을 때 돌아오는 결과.
* **체인/체이닝(Chain/Chaining)**: 문맥에 따라 다음을 의미합니다.
 * 사고 사슬(chain-of-thought) 프롬프팅에서는 논리적으로 이어지는 추론 단계.
 * LLM 시스템에서는 각 호출이 이전 호출의 응답에 의존하는 순차적인 LLM 호출.
* **예시(Exemplar)**: 원샷 또는 퓨샷 프롬프트에 들어가는 "예제".
 * 전통적인 머신러닝에서 말하는 "예제"(즉, "학습 데이터 한 건")와의 혼동을 피하기 위해 사용합니다.

## 환경 설정 -- 이 코드를 먼저 실행하세요!

In [ ]:
%%writefile requirements.txt
google-adk
google-genai
langchain_core
Langchain-google-genai

In [ ]:
%pip install -U -r requirements.txt | grep -v "Requirement already satisfied"

In [ ]:
# 설치한 패키지를 환경에서 사용할 수 있도록 커널을 자동으로 재시작합니다.
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

설치가 끝나면 새 패키지를 인식할 수 있도록 커널이 자동으로 재시작됩니다.

In [ ]:
PROJECT_ID = "YOUR-PROJECT-ID"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

from google import genai
from IPython.display import display, Markdown, clear_output

client = genai.Client(
    vertexai=True, project=PROJECT_ID, location='us-central1'
)

MODEL_ID='gemini-2.5-pro'

In [ ]:
chat_history= []

def chat(user_input):
    global chat_history


    user_message = genai.types.Content(
        role="user",
        parts=[genai.types.Part(text=user_input)]
    )

    chat_history.append(user_message)

    response = client.models.generate_content(
        model=MODEL_ID,
        contents = chat_history
    )

    # 결과에서 모델의 응답을 추출합니다.
    model_response_content = response.candidates[0].content
    model_text = model_response_content.parts[0].text

    display(Markdown(f"""**Gemini:**\n\n{model_text}"""))

    # 모델의 응답을 대화 기록 리스트에 덧붙입니다.
    chat_history.append(model_response_content)

## 결과물이 정해진 다단계 작업

계층적 프롬프팅(Hierarchical Prompting)은 대규모 언어 모델(LLM)이 생성하는 여러 부분으로 구성된 결과물의 일관성과 구조적 충실도를 보장하기 위한 고급 프롬프트 설계 전략입니다. 이 방법은 프로젝트 전체의 청사진을 지속적인 컨텍스트로 활용함으로써, 여러 번의 대화 턴을 거치는 동안 LLM이 초점을 잃거나 처음의 목표에서 벗어나는 경향을 효과적으로 억제합니다.

이 과정은 전체를 아우르는 포괄적인 작업을 정의하고 이를 번호가 매겨진 명확한 하위 작업(단계)의 순서로 구조화하는 것에서 시작합니다. 이 초기 계획이 곧 계층 구조가 됩니다. 실행 단계에서 사용자는 일련의 프롬프트를 반복해서 제출합니다. 첫 번째 프롬프트는 전체 구조 계획을 정의하고 모델에게 첫 번째 단계만 수행하도록 지시합니다. 이후의 모든 프롬프트에서는 대화 기록이 함께 덧붙여져 모델에 전달되고, 모델은 논리적으로 다음 단계를 수행합니다. 이미 수행한 단계와 남은 단계를 포함한 전체 범위를 계속 상기시킴으로써 LLM은 목표에 대한 완전한 맥락 지도를 유지하게 됩니다. 이러한 계층 구조의 포함 덕분에 최종 결과물은 구조적으로 탄탄하고, 주제적으로 일관되며, 처음에 정의한 장기 목표와 맞아떨어집니다.

또 다른 핵심 요소는 각 후속 단계가 이전 장(chapter)을 컨텍스트로 전달받는다는 점이며, 이를 통해 결과물이 이전 단계 위에 쌓여 나갑니다. 이는 후속 출력을 위해 대화 기록을 활용하는 것일 뿐입니다.

여기에 더해, 작업 목록 자체를 또 다른 프롬프트로 만들어 낼 수도 있습니다. 이 실습의 범위를 벗어나지만 구현하기는 상당히 쉽습니다. 또한 이런 유형의 작업은 단계별로 서로 다른 모델이 더 적합할 수 있는 대표적인 사례이기도 합니다.

대화 기록은 빈 리스트로 초기화하며, 이 주제에 잘 맞는 예시로 책 쓰기를 사용합니다.

In [ ]:
chat_history = []
chat("""달에 간 최초의 코끼리에 대한 이야기를 쓸 거예요. 과학자는 메이브 오코넬(Dr. Maeve O'Connell) 박사이고, 이야기 속 코끼리의 이름은 칼리스토(Callisto)입니다.
1장. 알맞은 코끼리 찾기
2장. 로켓 만들기
3장. 달로 가는 여정
4장. 파라스케보풀로스 크레이터 조사 https://en.wikipedia.org/wiki/Paraskevopoulos_(crater)
5장. 귀환 항해
6장. 영웅의 환영식

1장을 한국어로 써 주세요.
""")

In [ ]:
chat("""2장을 써 주세요.""")

In [ ]:
chat("""3장을 써 주세요.""")

In [ ]:
chat("""4장을 써 주세요.""")

In [ ]:
chat("""5장을 써 주세요.""")

In [ ]:
chat("""6장을 써 주세요.""")

전체 대화 기록을 확인해 봅시다. 대화 기록에서 책 전체를 추출할 수 있다는 점에서도 유용합니다.

In [ ]:
chat_history

대화의 일부로 이야기를 요약하게 할 수도 있습니다.

In [ ]:
chat("이야기를 한 편의 요약문(précis)으로 정리해 주세요.")

## LLM 입출력 안전을 위한 가디언 모델(Guardian Model)

**가디언 모델**(흔히 **가드레일** 또는 **안전 계층**이라고도 부릅니다)은 주 LLM과 함께 배치되어 사용자 입력과 모델 출력 양쪽의 데이터 흐름을 실시간으로 감시하고 검증하는 특수한 AI 시스템입니다. 주된 역할은 주 생성 모델을 재학습하거나 대규모로 파인튜닝하지 않고도 **안전성, 규정 준수, 품질 정책**을 강제하는 것입니다.

---

### 가디언 모델의 동작 방식

가디언 모델은 보통 주 모델보다 작고 계산 효율이 높은 보조 LLM으로 동작하며, 분류와 위험 탐지에 특화되도록 설계·학습됩니다. 이 모델은 통신 파이프라인의 두 지점에 끼어듭니다.

1.  **입력 필터링(사용자 프롬프트 검사):**
    * 사용자 프롬프트가 주 LLM에 도달하기 전에, 가디언 모델이 텍스트를 검사해 정책 위반을 탐지합니다.
    * 이를 통해 주 모델이 **탈옥(jailbreak) 시도**(안전 기능을 우회하도록 설계된 프롬프트), 악의적인 지시, 혐오 표현·폭력적 의도·불법 행위 같은 금지된 콘텐츠에 노출되는 것을 막습니다. 위험이 감지되면 요청을 차단하거나 사람 검토자에게 넘깁니다.

2.  **출력 검열(응답 검사):**
    * 주 LLM이 응답을 생성한 뒤, 사용자에게 반환되기 전에 가디언 모델이 그 출력을 검토합니다.
    * 유해하거나 편향된, 혹은 제한된 콘텐츠가 의도치 않게 생성되는 것을 잡아내는 데 매우 중요합니다. 또한 **환각(hallucination)** 여부(RAG 시스템에서 제공된 컨텍스트와 사실을 대조), 문맥 적합성, 전반적인 품질처럼 안전 외의 문제도 검사할 수 있습니다.

가디언 모델은 독립적이고 모듈화된 계층으로 동작하기 때문에, 고성능 범용 LLM은 생성 작업에 집중하고 책임 있는 AI 배포라는 중요한 과제는 전문화된 가디언이 담당하게 됩니다. 대표적인 예로 ShieldGemma, Meta의 Llama Guard, IBM의 Granite Guardian이 있습니다.

```mermaid
flowchart LR
A(["입력"])
A-->B["가디언"]
B-->C["부적절한 입력, 폐기"]
B-->D["LLM 질의"]
D-->E["가디언"]
E-->F["부적절한 출력, 폐기"]
E-->G(["출력"])
```

이를 위해 단일 호출(single shot) 프롬프트 핸들러를 작성하고, JSON을 반환하도록 유도해 봅니다. 모델에 강제하지 않으면 구조화된 응답을 출력한다는 보장이 없습니다.

In [ ]:
import json
from google.genai import types

def singleshot(user_input):

    user_message = genai.types.Content(
        role="user",
        parts=[genai.types.Part(text=user_input)]
    )

    system_instruction = '당신은 안전 검사 봇입니다. 아래 텍스트가 업무 환경에서 적절해 보이나요? {"appropriate":"yes"} 또는 {"appropriate":"no"} 형태의 JSON으로 응답하세요.'

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite", # Pro는 JSON이 아닌 응답을 보내는 경우가 있고, 가디언 모델에 Pro까지는 필요 없습니다.
        contents=user_input,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            max_output_tokens=100,
        ),
    )

    # 결과에서 모델의 응답을 추출합니다.
    model_response_content = response.candidates[0].content
    model_text = model_response_content.parts[0].text
    try:
        return json.loads(model_text)
    except:
        raise Exception(f"모델이 JSON을 반환하지 않았습니다!!!\n실제 반환값:\n{model_text}")

**참고:** 모델이 JSON 이외의 데이터를 생성할 수 있기 때문에 응답 처리가 **실패할 수도** 있습니다. 몇 번 반복 실행해 보면, 형식을 요청했음에도 모델이 항상 원하는 구조로 응답하지는 않는다는 것을 확인할 수 있습니다.

In [ ]:
user_input = "은행을 털어야 해"
for i in range(5):
    print(singleshot(user_input))

최신 모델들은 구조화된 응답을 강제하는 기능을 제공하는 경우가 많습니다. 이는 도구 호출(tool calling)에서 큰 이점이 있는데, 모델의 출력을 도구 함수 정의에 맞춰 구조화할 수 있기 때문입니다. (이를 위해서는 도구 호출 구조를 모델의 출력 형식 힌트로 전달할 수 있도록 PEP484 타입 힌트가 필요합니다.)

In [ ]:
from pydantic import BaseModel, Field

from typing import Literal

class SafetyResponse(BaseModel):
    """
    모델 응답 형식을 정의합니다.
    대응 형태: {"appropriate": "yes"} 또는 {"appropriate": "no"}
    """
    appropriate: Literal["yes", "no"] = Field(
        description="내용이 적절하면 반드시 'yes', 그렇지 않으면 'no'여야 합니다."
    )

def singleshot_structured(user_input):

    user_message = genai.types.Content(
        role="user",
        parts=[genai.types.Part(text=user_input)]
    )

    system_instruction = '당신은 안전 검사 봇입니다. 아래 텍스트가 업무 환경에서 적절해 보이나요? {"appropriate":"yes"} 또는 {"appropriate":"no"} 형태의 JSON으로 응답하세요.'

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=user_input,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            max_output_tokens=100,
            response_mime_type = "application/json",
            response_schema =  SafetyResponse,
        ),
    )

    # 결과에서 모델의 응답을 추출합니다.
    model_response_content = response.candidates[0].content
    model_text = model_response_content.parts[0].text
    try:
        return json.loads(model_text)
    except:
        raise Exception(f"모델이 JSON을 반환하지 않았습니다!!!\n실제 반환값:\n{model_text}")

이를 간단한 업무 흐름으로 구현하면 아래와 같이 가디언 모델을 적용할 수 있습니다.

In [ ]:
guardian1 = singleshot_structured(user_input)
if guardian1["appropriate"]=="no":
    print("어허, 마법의 주문을 말하지 않았군요")
else:
    response = singleshot(user_input)
    guardian2 = singleshot_structured(response)
    if guardian2["appropriate"]=="no":
        print("나쁜 LLM 같으니, 다시 상자 안으로")
    else:
        display(Markdown(response))

## 메타 프롬프팅(Meta Prompting)

메타 프롬프팅은 LLM을 이용해 자기 자신이나 다른 LLM이 사용할 지시 프롬프트를 자동으로 생성·최적화·개선하는 고급 프롬프트 엔지니어링 기법입니다. 이 방법은 LLM을 단순 실행자에서 자동화된 프롬프트 엔지니어로 격상시켜, 사용자는 상위 수준의 목표만 제시하고 AI가 작업을 성공적으로 수행하는 데 필요한 정밀하고 구조화된 지시문 세트를 구성하도록 합니다. 즉, 사용자의 의도와 최종 결과물 사이에 지능적인 추상화 계층을 만드는 전략입니다.

과정은 두 단계로 진행됩니다. 먼저 사용자가 출력될 프롬프트의 규칙과 구조 요건(어조, 형식, 필요한 섹션 등)을 기술한 상위 수준의 메타 프롬프트를 정의합니다. LLM은 이 메타 지시문을 처리해 페르소나 정의나 구조화된 서식(JSON, 마크다운 등) 같은 모범 사례를 반영한 생성 프롬프트를 만들어 냅니다. 이렇게 만들어진 고도로 최적화된 프롬프트를 사용해 최종 작업을 수행합니다.

이 기법은 고급 프롬프트 설계에 상당한 이점을 제공합니다. 반복 가능한 템플릿을 여러 작업에 강제해 일관성과 구조를 강화하고, 입력 맥락에 맞춰 프롬프트를 자동으로 조정하는 동적 적응을 가능하게 하며, 복잡한 멀티 에이전트 AI 아키텍처에서 지시문 생성을 자동화해 확장성을 높입니다. 결국 메타 프롬프팅은 지시문을 수작업으로 만드는 데서, 작업 완수 과정 전체를 LLM의 지능으로 최적화하는 쪽으로 초점을 옮깁니다.

이번에는 JSON 요구 사항 없이 간단한 단일 호출 핸들러를 다시 사용해 봅시다.

In [ ]:
def singleshot(user_input):

    user_message = genai.types.Content(
        role="user",
        parts=[genai.types.Part(text=user_input)]
    )

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=user_input,
        config=types.GenerateContentConfig(
            max_output_tokens=10000,
        ),
    )

    # 결과에서 모델의 응답을 추출합니다.
    model_response_content = response.candidates[0].content
    model_text = model_response_content.parts[0].text

    return model_text

In [ ]:
new_prompt = singleshot("달로 여행을 떠나는 코끼리에 대한 탄탄한 이야기 개요를 만들어 내기 위한 프롬프트를 한국어로 작성해 주세요")
display(Markdown("# 새로 생성된 프롬프트"))
display(Markdown(new_prompt))

In [ ]:
story_outline = singleshot(new_prompt)
display(Markdown("# 모델 응답"))
display(Markdown(story_outline))

<div class="alert alert-block alert-info">
<b>사이드 퀘스트:</b> 이제 자동 책 집필기를 만들 수 있는 도구를 모두 갖췄습니다. 아래 각 부분을 직접 구현해 보세요.


```mermaid

flowchart LR
USER(["사용자"])
USER-->Character["캐릭터 생성기"]
Character-->Idea["캐릭터를 바탕으로 이야기 아이디어 만들기"]
Idea-->Refine["프롬프트 개선하기"]
Refine-->Outline["구조화된 응답으로 장 목록을 JSON 배열 개요로 생성"]
Outline-->Chapter{"장 집필"}
Chapter--완료-->Done["책 출력"]
Chapter--미완료-->Chapter
```

</div>

## AI 패널 - 병렬 요청 - LangChain 활용

LangChain은 모듈 5에서 더 깊이 다룹니다.

좀 더 균형 잡힌 응답을 원한다면 어떻게 해야 할까요? 병렬 질의를 사용하면 하나의 프롬프트에 대해 서로 다른 관점을 강제하는 여러 요청을 보내고, 그 결과를 종합할 수 있습니다.

이 프롬프트 설계의 장점은 같은 문제에 대해 여러 관점을 얻을 수 있다는 것입니다. 패널리스트에 대한 설명이 구체적일수록 응답은 더 균형 잡히게 됩니다. 자문단에 일부러 반대 의견을 내는 역할(devil's advocate)을 추가해 볼 수도 있습니다.

```mermaid
flowchart LR
subgraph User
USER(["사용자"])
end
subgraph App
USER-->APP["앱"]
subgraph Personas
APP-->P1["LLM<br>페르소나: 앨런 튜링"]
APP-->P2["LLM<br>페르소나: 루스 베이더 긴즈버그"]
APP-->P3["LLM<br>페르소나: 로빈 윌리엄스"]
APP-->P4["LLM<br>페르소나: 마하트마 간디"]
APP-->P5["LLM<br>페르소나: 스티브 잡스"]
end

P1-->LLM1["페르소나별 응답"]
P2-->LLM1
P3-->LLM1
P4-->LLM1
P5-->LLM1
subgraph Aggregator
LLM1-->LLM2["요약 및 출처 표시:<br>종합 응답"]
end
end
LLM2-->USER
```

여기서는 LangChain의 병렬 실행기(parallel runner)를 사용해 패널리스트들을 병렬로 호출한 뒤, 그 결과를 간결한 요약으로 종합합니다. 두 개의 모델을 사용하며, 패널리스트에는 lite 모델을, 종합에는 flash 모델을 사용합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from vertexai.generative_models import HarmCategory, HarmBlockThreshold
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage, SystemMessage
from IPython.display import display, Markdown, clear_output

_='''safety_settings = {
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
} # '''

safety_settings = {
    "HARM_CATEGORY_DANGEROUS_CONTENT": "BLOCK_MEDIUM_AND_ABOVE",
    "HARM_CATEGORY_HARASSMENT": "BLOCK_MEDIUM_AND_ABOVE",
    "HARM_CATEGORY_HATE_SPEECH": "BLOCK_MEDIUM_AND_ABOVE",
    "HARM_CATEGORY_SEXUALLY_EXPLICIT": "BLOCK_MEDIUM_AND_ABOVE",
}

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=2,
    max_tokens=65535,
    max_retries=6,
    stop=None,
    top_p = 0.95,
    project=PROJECT_ID,
    safety_settings=safety_settings,
)
llm_agg = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=1,
    max_tokens=65535,
    max_retries=6,
    stop=None,
    top_p = 0.95,
    project=PROJECT_ID,
    safety_settings=safety_settings,
)

다음 절에서는 두 개의 프롬프트 템플릿으로 패널리스트의 흐름을 정의합니다. (더 많은 수로 확장할 수 있습니다.)

In [ ]:
def panel(user_input: str): #, max_tokens: int = None):
    """스트리밍 마크다운 출력으로 한 번의 대화 턴을 처리하는 함수."""
    global llm
    # 각 LLM에 사용할 시스템 프롬프트를 정의합니다.
    prompt_comedian = ChatPromptTemplate.from_messages([
        ("system", "당신은 프로 코미디언입니다. 사용자의 프롬프트에 재치 있고 웃긴 답변을 한국어로 하는 것이 목표입니다."),
        ("user", "{input}")
    ])

    prompt_technical = ChatPromptTemplate.from_messages([
        ("system", "당신은 진지하고 매우 상세한 기술 전문가입니다. 사용자의 프롬프트에 사실에 근거한 철저한 답변을 한국어로 하는 것이 목표입니다."),
        ("user", "{input}")
    ])

    # 각각의 체인을 생성합니다.
    chain_comedian = prompt_comedian | llm | StrOutputParser()
    chain_technical = prompt_technical | llm | StrOutputParser()
    # 두 체인을 병렬로 실행하도록 결합하고, 각 출력에 이름을 붙입니다.
    parallel_chain = RunnableParallel(
        comedian_response=chain_comedian,
        technical_response=chain_technical
    )

    # 이름 붙인 출력들을 참조하는 최종 종합 프롬프트를 정의합니다.
    aggregation_prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 유능한 어시스턴트입니다. 서로 다른 AI 페르소나들의 여러 응답이 주어집니다. 이 응답들을 검토해 하나의 일관되고 간결한 최종 답변을 한국어로 만들어 주세요. 각 응답의 핵심 내용과 어조를 살려 정보를 종합해야 하며, 두 응답을 단순히 나열해서는 안 됩니다."),
        ("user", "코미디언의 응답: {comedian_response}\n\n기술 전문가의 응답: {technical_response}")
    ])

    # 최종 종합 체인을 만듭니다.
    final_chain = {"comedian_response": chain_comedian, "technical_response": chain_technical} | aggregation_prompt | llm | StrOutputParser()
    try:
        # 같은 출력 셀을 계속 갱신하기 위해 임시 디스플레이 핸들을 사용합니다.
        output_handle = display(Markdown(f"챗봇: "), display_id=True)
        full_response = ""
        for chunk in final_chain.stream({
            #"chat_history": history,
            "input": user_input
        }):
            # 응답 청크를 누적합니다.
            full_response += chunk
            # 표시 중인 마크다운을 실시간으로 갱신합니다.
            output_handle.update(Markdown(f"**챗봇:**\n\n {full_response}"))

    except Exception as e:
        print(f"호출 중 오류가 발생했습니다: {e}")
        print("서버 연결 상태와 모델을 확인하세요.")

In [ ]:
display(panel("블록체인의 개념을 설명해 주세요."))

## 다른 프레임워크 - Google ADK 간단 소개

이 절에서는 간단한 ADK 에이전트를 살펴봅니다. 에이전트는 이 실습에서 다룬 모든 요소, 즉 시스템 프롬프트, 도구 호출, 메모리를 감싸는 래퍼입니다. 여기서는 또 다른 프레임워크를 소개하는 수준이며, 이 주제를 훨씬 자세히 다루는 별도의 과정이 있습니다.

에이전트를 정의합니다. 노트북과 일반 파이썬 코드 양쪽에서 동작하도록 만든 동기(synchronous) 방식 데모입니다.

CONFIGURATION 항목에 여러분의 PROJECT_ID를 입력하세요.

In [ ]:
import os
import asyncio
import nest_asyncio
from google.genai import Client, types
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

nest_asyncio.apply()

# 환경 변수를 설정합니다.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"

# ==============================================================================
# 3. 진단 테스트 (Runner를 시작하기 전에 연결 확인)
# ==============================================================================
print(f"--- {PROJECT_ID} 프로젝트로 API 연결 테스트 중 ---")
try:
    # Agent Platform에서 요구하는 특정 버전인 'gemini-2.5-pro'를 사용합니다.
    test_model_name = "gemini-2.5-pro"

    test_client = Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
    response = test_client.models.generate_content(
        model=test_model_name,
        contents="안녕하세요"
    )
    print("연결 테스트 통과! API가 정상적으로 응답합니다.")
except Exception as e:
    print(f"\n[치명적 오류] 모델 '{test_model_name}'로 API 연결에 실패했습니다.")
    print(f"오류 상세: {e}")
    # 연결할 수 없으면 여기서 중단합니다.
    raise e

# ==============================================================================
# 4. 에이전트 및 RUNNER 정의
# ==============================================================================
print("\n--- 에이전트 초기화 중 ---")

# 검증된 모델 이름으로 에이전트를 정의합니다.
root_agent = Agent(
    name="sync_runner_assistant",
    model="gemini-2.5-pro",
    instruction="당신은 동기 방식으로 실행되는 유능한 어시스턴트입니다. 한국어로 답변하세요.",
    description="동기 방식 스크립트를 위한 간단한 에이전트."
)

# Runner를 인스턴스화합니다.
runner = InMemoryRunner(
    agent=root_agent,
    app_name='my_sync_app',
)

user_sessions = {}

def get_or_create_session_id(user_id: str):
    if user_id not in user_sessions:
        async def create_new_session():
            session = await runner.session_service.create_session(
                app_name=runner.app_name,
                user_id=user_id
            )
            return session.id

        session_id = asyncio.run(create_new_session())
        user_sessions[user_id] = session_id
        return session_id
    else:
        return user_sessions[user_id]

def run_query_sync(user_id: str, message: str):
    session_id = get_or_create_session_id(user_id)

    content = types.Content(
        role='user',
        parts=[types.Part.from_text(text=message)]
    )

    print(f"사용자: {message}\n에이전트: ", end="")
    final_response = ""

    # Runner를 실행합니다.
    try:
        for event in runner.run(
            user_id=user_id,
            session_id=session_id,
            new_message=content,
        ):
            if event.content and event.content.parts and event.content.parts[0].text:
                text = event.content.parts[0].text
                print(text, end="", flush=True)
                final_response += text
    except Exception as e:
        print(f"\n\nRunner 실행 중 오류: {e}")

    print("\n" + "="*50 + "\n")
    return final_response

# ==============================================================================
# 5. 실행
# ==============================================================================
run_query_sync("sync_user_01", "용감한 펭귄에 대한 아주 짧은 이야기를 들려주세요.")
run_query_sync("sync_user_01", "방금 부탁한 이야기에 나온 동물은 무엇이었나요?")

**참고:** 표시되는 경고 메시지는 무시해도 괜찮습니다.